# GenAI Assignment 1 - training on Google Colab
1. **Runtime -> Change runtime type -> T4 GPU**.
2. Run the cells top to bottom. Each training cell saves its results to your Google Drive
   (`MyDrive/genai_outputs`) when it finishes, and stops with a red error if its stage failed.
3. **If Colab disconnects:** reconnect, run cells 1-4 again, then continue - finished stages are restored from Drive and skipped.
4. Keep this tab open (free Colab stops idle/closed sessions). Total time ~2.5-3.5 h on a T4.

In [ ]:
# 1) Google Drive for persistent results
from google.colab import drive
drive.mount('/content/drive')
DRIVE_OUT = '/content/drive/MyDrive/genai_outputs'
import os; os.makedirs(DRIVE_OUT, exist_ok=True)

In [ ]:
# 2) Code (clone, or pull the newest version) + dependencies
REPO_URL = "https://github.com/Abdullah8168/GenAI-Assignment1.git"
import os, subprocess
WORK = "/content/GenAI-Assignment1"
if os.path.exists(WORK):
    subprocess.run(["git", "-C", WORK, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, WORK], check=True)
os.chdir(WORK)
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)
!pip install -q optuna "mlflow>=2.17,<3" onnx onnxruntime gdown
import torch; print(torch.__version__, torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU - change runtime!")

In [ ]:
# 3) Datasets: FS2K from the official Google Drive link; Oxford-IIIT Pet downloads automatically in stage t1
import glob, zipfile
os.makedirs("/content/data", exist_ok=True)
if not glob.glob("/content/data/**/anno_train.json", recursive=True):
    !gdown 1saIMhQ3dc5_ftkfGmBPbCluRn_zy7QQp -O /content/data/FS2K.zip
    zipfile.ZipFile("/content/data/FS2K.zip").extractall("/content/data")
print(glob.glob("/content/data/**/anno_*.json", recursive=True))

In [ ]:
# 4) Environment, restore previous progress from Drive, helper to run one stage
import json, shutil
os.environ.update({"GENAI_PROFILE": "quick", "GENAI_DATA": "/content/data", "FS2K_ROOT": "/content/data",
                   "GENAI_OUT": f"{WORK}/outputs", "GENAI_WORKERS": "2"})
OUT = f"{WORK}/outputs"
DONE = f"{OUT}/tables/stage_minutes.json"
if os.path.exists(f"{DRIVE_OUT}/outputs.zip") and not os.path.exists(DONE):
    zipfile.ZipFile(f"{DRIVE_OUT}/outputs.zip").extractall(OUT)
print("finished stages:", json.load(open(DONE)) if os.path.exists(DONE) else "none yet")

def save_to_drive():
    shutil.make_archive("/content/outputs", "zip", OUT)
    shutil.copy("/content/outputs.zip", f"{DRIVE_OUT}/outputs.zip")
    print("saved outputs.zip to Drive")

def run_stage(stage):
    get_ipython().system(f"python -u run_all.py --stages {stage} --skip-done 2>&1 | tee -a /content/train_log.txt")
    done = json.load(open(DONE)) if os.path.exists(DONE) else {}
    if stage not in done:
        raise RuntimeError(f"Stage {stage} FAILED - copy the error above and send it to Claude")
    save_to_drive()

In [ ]:
# 5) Task 1 - universal denoising autoencoder (~30-40 min)
run_stage("t1")

In [ ]:
# 6) Task 2 - classifier + specialists (~45-60 min)
run_stage("t2")

In [ ]:
# 7) Task 3 - soft mixture of experts (~30-40 min)
run_stage("t3")

In [ ]:
# 8) Task 4 - face-to-sketch cGAN (~50-70 min)
run_stage("t4")

In [ ]:
# 9) ONNX export + verification, final save
run_stage("export")
shutil.copy("/content/train_log.txt", f"{OUT}/train_log.txt")
save_to_drive()
print(open(f"{OUT}/tables/onnx_verification.csv").read())
print(r"Download MyDrive/genai_outputs/outputs.zip and unzip it into D:\genai-assignment1\outputs")